# 05_Baseline_TFIDF_Models

## Objective

The goal of this notebook is to develop baseline machine learning models for early sepsis prediction using clinical notes.

This notebook will:

- load the cleaned clinical notes dataset
- split the data into training, validation, and test sets
- convert clinical notes into TF-IDF features
- train Logistic Regression and XGBoost classifiers
- evaluate model performance using classification metrics

The resulting models will serve as baseline benchmarks for comparison with transformer-based NLP models in later stages of the project.

In [1]:
# Code Starter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"
OUTPUT_PATH = "../outputs/"

## Step 1: Import Libraries

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

## Step 2: Load Dataset

In [6]:
clean_notes = pd.read_csv(PROCESSED_PATH + "clean_notes.csv")

print(clean_notes.shape)
clean_notes.head()

(36395, 5)


,SUBJECT_ID,HADM_ID,ICUSTAY_ID,SEPSIS_LABEL,CLEAN_TEXT
0,3,145834.0,211552,1,10:23 pm chest (portable ap) clip # reason: pl...
1,4,185777.0,294638,0,0500 general: pt in to ew from home with c/o f...
2,6,107064.0,228232,0,2230-0700 recieved pt from pacu following lr k...
3,9,150750.0,220597,0,respiratory care: pt. intubated in ew for airw...
4,11,194540.0,229441,0,2:49 pm cta head w&w/o c & recons clip # reaso...


## Step 3: Define Features and Labels

In [9]:
X = clean_notes["CLEAN_TEXT"]

y = clean_notes["SEPSIS_LABEL"]

## Step 4: Train / Validation / Test Split

In [16]:
# First Split 
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# Second Split
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# Verify Size of Splits
print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Train: 25476
Validation: 5459
Test: 5460


## Step 5: TF-IDF Feature Engineering

In [19]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), stop_words="english")

# Fit on Training Data Only
X_train_tfidf = tfidf.fit_transform(X_train)

X_val_tfidf = tfidf.transform(X_val)

X_test_tfidf = tfidf.transform(X_test)

# Check Feature Matrix
print(X_train_tfidf.shape)

(25476, 10000)


## Step 6: Logistic Regression

In [ ]:
lr_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)

# Train
lr_model.fit(X_train_tfidf, y_train)

# Validation Predictions
lr_val_pred = lr_model.predict(X_val_tfidf)

lr_val_prob = lr_model.predict_proba(X_val_tfidf)[:,1]

### Evaluate Logistic Regression

In [31]:
acc_log = accuracy_score(y_val, lr_val_pred)
print("Accuracy:", acc_log)

prec_log = precision_score(y_val, lr_val_pred)
print("Precision:", prec_log)

rec_log = recall_score(y_val, lr_val_pred)
print("Recall:", rec_log)
      
f1_log = f1_score(y_val, lr_val_pred)
print("F1:", f1_log)

roc_log = roc_auc_score(y_val, lr_val_prob)
print("ROC-AUC:", roc_log)

Accuracy: 0.8624290163033522
Precision: 0.43308550185873607
Recall: 0.7677100494233937
F1: 0.5537730243612596
ROC-AUC: 0.9124378133102264


## Step 7: XGBoost

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

# Train
xgb_model.fit(X_train_tfidf, y_train)

# Validation Prediction
xgb_val_pred = xgb_model.predict(X_val_tfidf)

xgb_val_prob = xgb_model.predict_proba(X_val_tfidf)[:,1]


### Evaluate XGBoost

In [34]:
acc_xgb = accuracy_score(y_val, xgb_val_pred)
print("Accuracy:", acc_xgb)

prec_xgb = precision_score(y_val, xgb_val_pred)
print("Precision:", prec_xgb)

rec_xgb = recall_score(y_val, xgb_val_pred)
print("Recall:", rec_xgb)
      
f1_xgb = f1_score(y_val, xgb_val_pred)
print("F1:", f1_xgb)

roc_xgb = roc_auc_score(y_val, xgb_val_prob)
print("ROC-AUC:", roc_xgb)

Accuracy: 0.9221469133540942
Precision: 0.7321428571428571
Recall: 0.4728171334431631
F1: 0.5745745745745746
ROC-AUC: 0.9082458565974593


## Step 8: Compare Both Models

In [40]:
results = pd.DataFrame({
    
    "Model" : ["Logistic Regression", "XGBoost"],
    "Accuracy" : [acc_log, acc_xgb],
    "Precision" : [prec_log, prec_xgb],
    "Recall" : [rec_log, rec_xgb],
    "ROC_AUC" : [roc_log, roc_xgb],
    "F1" : [f1_log, f1_xgb]
    
})

results

,Model,Accuracy,Precision,Recall,ROC_AUC,F1
0,Logistic Regression,0.862429,0.433086,0.767710,0.912438,0.553773
1,XGBoost,0.922147,0.732143,0.472817,0.908246,0.574575


## Step 9: Save Evaluation Results and Model

### Save Model Comparison Table

In [46]:
results.to_csv(
    OUTPUT_PATH + "baseline_model_results.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.


### Save Logistic Regression Confusion Matrix

In [49]:
lr_cm = confusion_matrix(y_val, lr_val_pred)

lr_cm_df = pd.DataFrame(
    lr_cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

lr_cm_df.to_csv(
    OUTPUT_PATH + "logistic_regression_confusion_matrix.csv"
)

lr_cm_df

,Predicted 0,Predicted 1
Actual 0,4242,610
Actual 1,141,466


### Save XGBoost Confusion Matrix

In [53]:
xgb_cm = confusion_matrix(y_val, xgb_val_pred)

xgb_cm_df = pd.DataFrame(
    xgb_cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

xgb_cm_df.to_csv(
    OUTPUT_PATH + "xgboost_confusion_matrix.csv"
)

xgb_cm_df

,Predicted 0,Predicted 1
Actual 0,4747,105
Actual 1,320,287


### Save Logistic Regression Classification Report

In [56]:
lr_report = classification_report(
    y_val,
    lr_val_pred,
    output_dict=True
)

lr_report_df = pd.DataFrame(lr_report).transpose()

lr_report_df.to_csv(
    OUTPUT_PATH + "logistic_regression_classification_report.csv"
)

lr_report_df

,precision,recall,f1-score,support
0,0.967830,0.874279,0.918679,4852.000000
1,0.433086,0.767710,0.553773,607.000000
accuracy,0.862429,0.862429,0.862429,0.862429
macro avg,0.700458,0.820994,0.736226,5459.000000
weighted avg,0.908371,0.862429,0.878104,5459.000000


### Save XGBoost Classification Report

In [59]:
xgb_report = classification_report(
    y_val,
    xgb_val_pred,
    output_dict=True
)

xgb_report_df = pd.DataFrame(xgb_report).transpose()

xgb_report_df.to_csv(
    OUTPUT_PATH + "xgboost_classification_report.csv"
)

xgb_report_df

,precision,recall,f1-score,support
0,0.936846,0.978359,0.957153,4852.000000
1,0.732143,0.472817,0.574575,607.000000
accuracy,0.922147,0.922147,0.922147,0.922147
macro avg,0.834495,0.725588,0.765864,5459.000000
weighted avg,0.914085,0.922147,0.914613,5459.000000


### Save the Trained Models

In [62]:
MODEL_PATH = "../models/"

os.makedirs(MODEL_PATH, exist_ok=True)

import joblib

joblib.dump(
    lr_model,
    MODEL_PATH + "logistic_regression.pkl"
)

joblib.dump(
    xgb_model,
    MODEL_PATH + "xgboost.pkl"
)

print("Models saved successfully.")

Models saved successfully.


## Summary

In this notebook, baseline machine learning models were developed to predict sepsis using clinical notes collected within the first 24 hours of ICU admission.

The cleaned clinical note dataset was loaded and divided into training, validation, and testing subsets using a stratified 70% / 15% / 15% split to preserve the original class distribution. The clinical text was transformed into numerical features using the TF-IDF (Term Frequency–Inverse Document Frequency) technique, resulting in a high-dimensional feature representation suitable for machine learning algorithms.

Two baseline models were trained and evaluated:

* Logistic Regression
* XGBoost

Model performance was assessed using Accuracy, Precision, Recall, F1 Score, and ROC-AUC metrics. Both models demonstrated strong predictive performance, indicating that meaningful information related to sepsis is present within the first 24 hours of ICU clinical documentation.

The results revealed different prediction behaviors between the two models. Logistic Regression achieved stronger recall performance, identifying a larger proportion of sepsis cases, while XGBoost achieved higher precision and overall accuracy. This highlights the trade-off between detecting as many sepsis cases as possible and minimizing false positive predictions.

Overall, the baseline experiments establish a strong reference point for the remainder of the project. The results confirm that traditional machine learning approaches combined with TF-IDF representations can effectively utilize clinical notes for sepsis prediction. These baseline models will be used for comparison against transformer-based approaches, including ClinicalBERT, in subsequent stages of the research.

All model outputs, evaluation metrics, confusion matrices, classification reports, and trained model files were saved for future analysis and comparison.
